In [ ]:
import pandas as pd

DATA = '../data/case-study/processed'

def calculate_hrv_metrics(ibi_series):
    sdnn = ibi_series.std()
    ibi_diff = ibi_series.diff().dropna()
    rmssd = (ibi_diff ** 2).mean() ** 0.5
    return sdnn, rmssd

# process all sessions
files = [('hr', 'ibi', 'hrv')] + [(f'hr_{s:02d}', f'ibi_{s:02d}', f'hrv_{s:02d}') for s in [1, 2, 3]]

for hr_name, ibi_name, hrv_name in files:
    hr_data = pd.read_csv(f'{DATA}/{hr_name}.csv')
    ibi_data = pd.read_csv(f'{DATA}/{ibi_name}.csv')

    combined_data = pd.merge(hr_data, ibi_data, on=['reltime', 'datetime', 'iSensor'], suffixes=('_hr', '_ibi'))
    valid_ibi_data = combined_data[combined_data['ibi'] > 0]

    sdnn, rmssd = calculate_hrv_metrics(valid_ibi_data['ibi'])

    hrv_data = valid_ibi_data[['reltime', 'datetime']].copy()
    hrv_data['sdnn'] = sdnn
    hrv_data['rmssd'] = rmssd

    hrv_data.to_csv(f'{DATA}/{hrv_name}.csv', index=False)
    print(f'{hrv_name}: SDNN = {sdnn:.2f} ms, RMSSD = {rmssd:.2f} ms')